# Vector Databases in Production

Deep dive into vector databases — architecture, indexing algorithms, metadata filtering, and performance benchmarking for production RAG systems.

**Topics:** ChromaDB, Pinecone, pgvector, FAISS, metadata filtering, namespace isolation, ANN algorithms

## 1. ChromaDB — Collections, Metadata, and Filtering

In [ ]:
import chromadb
from chromadb.utils import embedding_functions
from chromadb.config import Settings

# Persistent client (survives restarts)
chroma = chromadb.PersistentClient(
    path='./chroma_production',
    settings=Settings(anonymized_telemetry=False),
)

# Create collection with custom embedding function
ef = embedding_functions.SentenceTransformerEmbeddingFunction(model_name='all-MiniLM-L6-v2')

collection = chroma.get_or_create_collection(
    name='knowledge_base',
    embedding_function=ef,
    metadata={
        'hnsw:space': 'cosine',         # similarity metric
        'hnsw:construction_ef': 200,    # build quality (higher = better but slower)
        'hnsw:M': 32,                   # connections per node (16-64 recommended)
    },
)

# Upsert with rich metadata for filtering
collection.upsert(
    ids=['doc_001', 'doc_002', 'doc_003'],
    documents=[
        'Transformers use self-attention mechanisms.',
        'RAG combines retrieval with generation.',
        'BERT is pre-trained with masked language modeling.',
    ],
    metadatas=[
        {'topic': 'transformers', 'year': 2017, 'difficulty': 'advanced', 'lang': 'en'},
        {'topic': 'rag', 'year': 2020, 'difficulty': 'intermediate', 'lang': 'en'},
        {'topic': 'bert', 'year': 2018, 'difficulty': 'advanced', 'lang': 'en'},
    ],
)

# Metadata filtering (combined with semantic search)
results = collection.query(
    query_texts=['What is self-attention?'],
    n_results=5,
    where={'$and': [{'year': {'$gte': 2018}}, {'difficulty': {'$eq': 'advanced'}}]},
    where_document={'$contains': 'pre-trained'},  # filter on document text
)

print(f'Collection: {collection.name}')
print(f'Documents: {collection.count()}')
print()
print('ChromaDB filter operators:')
for op, meaning in [('$eq','equals'),('$ne','not equals'),('$gt/$gte','greater than'),('$lt/$lte','less than'),('$in','in list'),('$and/$or','logical')]:
    print(f'  {op:<12}: {meaning}')

## 2. FAISS — Flat vs IVF vs HNSW Indexes

In [ ]:
import faiss
import numpy as np
import time

def benchmark_faiss_indexes(n_vectors: int = 100_000, dim: int = 384, top_k: int = 10):
    """Compare FAISS index types on speed and accuracy."""
    np.random.seed(42)
    data = np.random.randn(n_vectors, dim).astype('float32')
    faiss.normalize_L2(data)
    query = np.random.randn(1, dim).astype('float32')
    faiss.normalize_L2(query)
    
    results = {}
    
    # Flat (exact): O(n) search, 100% recall — use for <1M vectors
    flat_index = faiss.IndexFlatIP(dim)  # Inner Product = cosine for normalized vectors
    flat_index.add(data)
    t0 = time.perf_counter()
    exact_D, exact_I = flat_index.search(query, top_k)
    results['Flat (exact)'] = {'time_ms': (time.perf_counter()-t0)*1000, 'recall': 1.0, 'mem_mb': n_vectors*dim*4/1e6}
    
    # IVF: cluster-based, O(n/nlist) search — good for 1M-10M vectors
    nlist = 100
    quantizer = faiss.IndexFlatIP(dim)
    ivf_index = faiss.IndexIVFFlat(quantizer, dim, nlist, faiss.METRIC_INNER_PRODUCT)
    ivf_index.train(data)
    ivf_index.add(data)
    ivf_index.nprobe = 10  # number of clusters to search (higher = more accurate but slower)
    t0 = time.perf_counter()
    ivf_D, ivf_I = ivf_index.search(query, top_k)
    ivf_recall = len(set(exact_I[0]) & set(ivf_I[0])) / top_k
    results['IVF (nprobe=10)'] = {'time_ms': (time.perf_counter()-t0)*1000, 'recall': ivf_recall, 'mem_mb': n_vectors*dim*4/1e6}
    
    # HNSW: graph-based, O(log n) search — best for real-time serving
    hnsw_index = faiss.IndexHNSWFlat(dim, 32)  # M=32 connections
    hnsw_index.hnsw.efConstruction = 200
    hnsw_index.hnsw.efSearch = 128
    hnsw_index.add(data)
    t0 = time.perf_counter()
    hnsw_D, hnsw_I = hnsw_index.search(query, top_k)
    hnsw_recall = len(set(exact_I[0]) & set(hnsw_I[0])) / top_k
    results['HNSW (M=32)'] = {'time_ms': (time.perf_counter()-t0)*1000, 'recall': hnsw_recall, 'mem_mb': n_vectors*dim*4*2/1e6}
    
    return results

print('FAISS index comparison:')
print(f'{"Index":<18} {"Time (ms)":<12} {"Recall@10":<12} {"Memory (MB)":<14} {"Use case"}')
print('-' * 75)
simulated = [
    ('Flat (exact)', 85.0, 1.00, 144, '<500K vectors, need exact results'),
    ('IVF (nprobe=10)', 2.1, 0.94, 144, '1M-50M vectors, accept ~5% recall loss'),
    ('HNSW (M=32)', 0.8, 0.98, 290, 'Real-time serving, high recall'),
    ('IVF-PQ (compressed)', 1.5, 0.88, 18, '10M+ vectors, memory constrained'),
]
for name, t, recall, mem, use in simulated:
    print(f'{name:<18} {t:<12.1f} {recall:<12.2f} {mem:<14} {use}')

## 3. pgvector — SQL + Vectors in PostgreSQL

In [ ]:
import psycopg2
from pgvector.psycopg2 import register_vector
import numpy as np

SETUP_SQL = """
-- Enable pgvector extension
CREATE EXTENSION IF NOT EXISTS vector;

-- Documents table with vector column
CREATE TABLE IF NOT EXISTS documents (
    id          SERIAL PRIMARY KEY,
    content     TEXT NOT NULL,
    embedding   VECTOR(1536),  -- OpenAI text-embedding-3-small dimension
    source      TEXT,
    topic       TEXT,
    created_at  TIMESTAMPTZ DEFAULT NOW(),
    metadata    JSONB DEFAULT '{}'
);

-- IVFFlat index (train after inserting data)
CREATE INDEX IF NOT EXISTS documents_embedding_idx
    ON documents USING ivfflat (embedding vector_cosine_ops)
    WITH (lists = 100);  -- sqrt(n_rows) is a good default
"""

SEARCH_SQL = """
SELECT id, content, source, topic,
       1 - (embedding <=> %s::vector) AS cosine_similarity
FROM documents
WHERE topic = %s           -- metadata filter (uses B-tree index)
  AND created_at > NOW() - INTERVAL '30 days'
ORDER BY embedding <=> %s::vector  -- cosine distance (<=> operator)
LIMIT 5;
"""

def pgvector_search(conn, query_embedding: np.ndarray, topic: str, top_k: int = 5):
    """Hybrid SQL + vector search in PostgreSQL."""
    with conn.cursor() as cur:
        embedding_list = query_embedding.tolist()
        cur.execute(SEARCH_SQL, (embedding_list, topic, embedding_list))
        return cur.fetchall()

print('pgvector advantages:')
advantages = [
    'ACID transactions — vector updates are consistent with other data',
    'SQL joins — combine vector search with relational filtering',
    'Existing infra — if you already run Postgres, no new DB',
    'JSONB metadata — flexible schema alongside vectors',
    'Row-level security — multi-tenant isolation via Postgres RLS',
]
for adv in advantages:
    print(f'  ✓ {adv}')
print()
print('pgvector operators:')
for op, name, use in [('<=>','Cosine distance','Semantic similarity'),('<->','L2 distance','Euclidean search'),('<#>','Negative inner product','Dot product')]:
    print(f'  {op:<6}: {name:<22} — {use}')

## 4. Metadata Filtering Strategies

In [ ]:
from dataclasses import dataclass
from typing import Any

@dataclass
class FilterStrategy:
    name: str
    approach: str
    pros: list[str]
    cons: list[str]

# Common metadata filtering patterns in vector DBs
strategies = [
    FilterStrategy(
        'Pre-filter (before ANN)',
        'Filter documents first, then run vector search on subset',
        ['Fast when filter is selective (few docs match)', 'No wasted vector computation'],
        ['Poor recall when filter matches many docs', 'Can miss relevant docs'],
    ),
    FilterStrategy(
        'Post-filter (after ANN)',
        'Run vector search, then filter results by metadata',
        ['High recall — sees all candidate vectors', 'Simple implementation'],
        ['Wasteful if most results are filtered out', 'May return fewer than k results'],
    ),
    FilterStrategy(
        'In-filter (HNSW with filter)',
        'Filter applied during graph traversal',
        ['Best balance of speed and recall', 'Used by Qdrant, Weaviate'],
        ['Complex to implement', 'Not supported by all DBs'],
    ),
]

# Namespace/tenant isolation patterns
def namespace_key(tenant_id: str, collection: str) -> str:
    """Isolate data by tenant using namespaced collection names."""
    return f'{tenant_id}:{collection}'

tenant_designs = {
    'Namespace': 'Separate namespace per tenant in same DB (Pinecone)',
    'Collection': 'Separate collection per tenant (ChromaDB)',
    'Metadata filter': 'Single collection, filter by tenant_id (simpler but slower)',
    'Separate DB': 'Full database per tenant (max isolation, highest cost)',
}

print('Metadata filtering strategies:')
for s in strategies:
    print(f'  {s.name}:')
    print(f'    Approach: {s.approach}')
    print(f'    Pro: {s.pros[0]}')
    print(f'    Con: {s.cons[0]}')
    print()
print('Multi-tenant isolation options:')
for design, desc in tenant_designs.items():
    print(f'  {design:<18}: {desc}')

## 5. Benchmarking Vector Database Performance

In [ ]:
import numpy as np
import time
from dataclasses import dataclass

@dataclass
class VectorDBBenchmark:
    db_name: str
    n_vectors: int
    insert_throughput_kps: float  # thousands of vectors per second
    query_latency_p50_ms: float
    query_latency_p99_ms: float
    recall_at_10: float
    qps: float  # queries per second

def measure_latency_percentiles(
    search_fn, queries: np.ndarray, n_runs: int = 100
) -> tuple[float, float]:
    """Measure p50 and p99 query latency."""
    latencies = []
    for q in queries[:n_runs]:
        t0 = time.perf_counter()
        search_fn(q.reshape(1, -1))
        latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.percentile(latencies, 50)), float(np.percentile(latencies, 99))

# Benchmark comparison (simulated for 1M 1536-dim vectors)
benchmarks = [
    VectorDBBenchmark('Pinecone (managed)', 1_000_000, 5.0,  1.8,  12.0, 0.98, 800),
    VectorDBBenchmark('Qdrant (on-prem)',   1_000_000, 12.0, 2.1,  15.0, 0.97, 650),
    VectorDBBenchmark('Weaviate (managed)', 1_000_000, 8.0,  3.5,  25.0, 0.97, 500),
    VectorDBBenchmark('ChromaDB (local)',   1_000_000, 3.0,  8.0,  80.0, 0.95, 120),
    VectorDBBenchmark('pgvector (IVFFlat)', 1_000_000, 4.0,  5.0,  45.0, 0.93, 200),
    VectorDBBenchmark('FAISS (HNSW)',       1_000_000, 30.0, 0.8,   5.0, 0.98, 2000),
]

print(f'{"DB":<25} {"Insert (k/s)":<14} {"p50 (ms)":<10} {"p99 (ms)":<10} {"Recall":<8} {"QPS"}')
print('-' * 80)
for b in sorted(benchmarks, key=lambda x: x.query_latency_p50_ms):
    print(f'{b.db_name:<25} {b.insert_throughput_kps:<14.1f} {b.query_latency_p50_ms:<10.1f} {b.query_latency_p99_ms:<10.1f} {b.recall_at_10:<8.2f} {b.qps:.0f}')
print()
print('Recommendations:')
print('  < 100K docs, prototyping : ChromaDB (simple, local)')
print('  100K-10M, managed cloud  : Pinecone or Qdrant Cloud')
print('  Already using Postgres   : pgvector')
print('  Max performance, on-prem : FAISS + custom serving')

## Exercises

1. **Index selection:** Build a benchmark harness that: (1) generates 500K random 768-dim vectors, (2) builds Flat, IVF-PQ, and HNSW indexes, (3) measures recall@10 and QPS for each. Plot the recall-speed tradeoff.

2. **Hybrid filter:** Implement a combined keyword + vector search in pgvector using PostgreSQL's full-text search (`tsvector`) for the keyword component and `<=>` for the vector component. Use RRF to merge results.

3. **Multi-tenant isolation:** Build a `TenantVectorStore` class that wraps ChromaDB and: stores each tenant's documents in a separate collection, enforces tenant isolation in all operations, and provides admin stats (docs per tenant, storage size).